# L12 failure: checkpoint rebalance audit

Attach **Tomahawk_G2_Corrected_Input_v2.zip** (or its extracted source files) and **Tomahawk_G2_Run_2_v2_checkpoint.zip** (or its extracted files) to the Kaggle notebook. Enable Internet if `desc-opt==0.17.3` is not already installed. Run the code cell.

This reads two saved equilibria and the verified MGRID. It performs no optimization. The small result ZIP is written to `/kaggle/working/Tomahawk_G2_L12_Audit.zip`. The notebook reports an error inside that ZIP even when an input is missing or a dependency fails.

## Verified checkpoint finding

This notebook was run read-only against the exact saved checkpoint and MGRID with DESC 0.17.3. Both states have the same boundary to numerical precision and the same TF/PF currents.

| State | Volume force RMS | Normal field maximum | Pressure mismatch maximum |
|:--|--:|--:|--:|
| L12 free | 0.070695 | 0.016171 | 0.029106 |
| L16 fixed, force rebalanced | 0.0000140 | 0.079930 | 0.020705 |

All figures use the same evaluation grids. The declared gate for each column is 0.001. A bounded PF-only normal-field fit at the balanced surface returns all six groups at 50% nominal. Extending the diagnostic range to 0% leaves a normal maximum of 0.068360; even an unbounded mathematical fit has normal maximum 0.016682. Thus coil-current retuning alone does not repair this saved equilibrated surface. These are screens at a fixed surface and fixed pressure/iota/flux profile; they do not establish that every redesigned surface or profile fails.


In [ ]:
"""Read-only L12/L16 checkpoint comparison for Kaggle (DESC 0.17.3).

Attach the corrected G2 input ZIP (or its extracted files) and the Run 2 v2
checkpoint ZIP (or its extracted files) as Kaggle input datasets, then run.
The script evaluates saved states; it never calls an optimizer or saves an
equilibrium. Output is /kaggle/working/Tomahawk_G2_L12_Audit.zip.
"""

from __future__ import annotations

import contextlib
import hashlib
import importlib.metadata
import io
import json
import os
from pathlib import Path
from pathlib import PurePosixPath
import shutil
import subprocess
import sys
import traceback
import zipfile


MGRID = "mgrid_tomahawk_16T_TF_PF_16FP_FULL_RECOMPUTED.nc"
MGRID_SHA256 = "afb07089205aa3d2cdfa03d53b48298ae5092f23c0e70db72938c21f59a78013"
ENGINE_SHA256 = "6bf8ea82626f6c3d485fef673ea12de6d4a5074189188bb059f8c71371fae043"
SOURCE_MANIFEST_SHA256 = "86b9481410b1b0cacf00a3b20733827722742d4843b371987de92cbb0191bb8e"
CHECKPOINT_SHA256 = "1535aee65adc1db7411ae1fc2a33622ae34ca522ce3ca6dce40b461c02ccda85"
PF_LOWER = 0.5
BOUNDARY_GATE = 1.0e-3
FORCE_GATE = 1.0e-3

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
OUT_DIR = WORK_ROOT / "l12_readonly_audit"
OUT_ZIP = WORK_ROOT / "Tomahawk_G2_L12_Audit.zip"


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def json_write(name: str, value: dict) -> None:
    (OUT_DIR / name).write_text(json.dumps(value, indent=2, allow_nan=False) + "\n")


def ensure_desc() -> None:
    os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
    os.environ.setdefault("JAX_ENABLE_X64", "True")
    try:
        installed = importlib.metadata.version("desc-opt")
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != "0.17.3":
        print(f"Installing desc-opt==0.17.3 (found {installed!r})", flush=True)
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "desc-opt==0.17.3"],
            check=True,
        )
    if importlib.metadata.version("desc-opt") != "0.17.3":
        raise RuntimeError("DESC 0.17.3 is required for comparison with the saved run")


def zip_matches(path: Path, suffix: str) -> list[str]:
    try:
        with zipfile.ZipFile(path) as archive:
            return [n for n in archive.namelist() if n.endswith(suffix)]
    except zipfile.BadZipFile:
        return []


def unique_choice(matches: list, description: str):
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected exactly one {description} under {INPUT_ROOT}; found {len(matches)}: "
            f"{[str(x) for x in matches[:8]]}"
        )
    return matches[0]


def prepare_mgrid() -> Path:
    direct = [p for p in INPUT_ROOT.rglob(MGRID) if p.is_file()]
    zipped = []
    if not direct:
        for path in INPUT_ROOT.rglob("*.zip"):
            for member in zip_matches(path, MGRID):
                if PurePosixPath(member).name == MGRID:
                    zipped.append((path, member))
    if direct:
        source = unique_choice(direct, "extracted MGRID")
        print(f"Using MGRID: {source}", flush=True)
        target = source
    else:
        source, member = unique_choice(zipped, "MGRID in a source ZIP")
        target = OUT_DIR / MGRID
        print(f"Extracting {member} from {source}", flush=True)
        with zipfile.ZipFile(source) as archive, archive.open(member) as src, target.open("wb") as dest:
            shutil.copyfileobj(src, dest, length=8 * 1024 * 1024)
    actual = sha256_file(target)
    if actual != MGRID_SHA256:
        raise ValueError(f"Wrong MGRID SHA256: {actual} (expected {MGRID_SHA256})")
    print("Exact 16 T MGRID hash verified", flush=True)
    return target


def prepare_checkpoint() -> tuple[dict[str, Path], dict, dict]:
    required = {
        "l12_free": "steps/L12/free_equilibrium.h5",
        "l16_fixed": "steps/L16/fixed_equilibrium.h5",
        "l12_metrics": "steps/L12/metrics.json",
        "run_status": "run_status.json",
    }
    direct = list(INPUT_ROOT.rglob(required["l12_free"]))
    candidates = []
    if not direct:
        for path in INPUT_ROOT.rglob("*.zip"):
            if zip_matches(path, required["l12_free"]):
                candidates.append(path)
        # A small Claude handoff ZIP can contain the checkpoint ZIP as a member.
        if not candidates:
            for path in INPUT_ROOT.rglob("*.zip"):
                for member in zip_matches(path, "Tomahawk_G2_Run_2_v2_checkpoint.zip"):
                    candidate = OUT_DIR / "embedded_checkpoint.zip"
                    with zipfile.ZipFile(path) as archive, archive.open(member) as src, candidate.open("wb") as dest:
                        shutil.copyfileobj(src, dest)
                    candidates.append(candidate)
    files = {}
    if direct:
        root = unique_choice(direct, "extracted L12 free equilibrium").parents[2]
        for key, relative in required.items():
            source = root / relative
            if not source.is_file():
                raise FileNotFoundError(f"Checkpoint is missing {source}")
            files[key] = source
    else:
        source = unique_choice(candidates, "Run 2 v2 checkpoint ZIP")
        if source.name == "Tomahawk_G2_Run_2_v2_checkpoint.zip":
            actual = sha256_file(source)
            if actual != CHECKPOINT_SHA256:
                raise ValueError(f"Wrong checkpoint SHA256: {actual}")
        with zipfile.ZipFile(source) as archive:
            for key, relative in required.items():
                member = unique_choice(
                    [n for n in archive.namelist() if n.endswith(relative)], relative
                )
                dest = OUT_DIR / Path(relative).name
                with archive.open(member) as src, dest.open("wb") as out:
                    shutil.copyfileobj(src, out)
                files[key] = dest
        print(f"Loaded saved equilibria from {source}", flush=True)
    metrics = json.loads(files["l12_metrics"].read_text())
    status = json.loads(files["run_status"].read_text())
    if status.get("engine_sha256") != ENGINE_SHA256:
        raise ValueError("Checkpoint was produced by a different G2 engine")
    if status.get("source_manifest_sha256") != SOURCE_MANIFEST_SHA256:
        raise ValueError("Checkpoint was produced by a different G2 source manifest")
    if metrics.get("resolution") != [12, 12, 3] or metrics.get("scenario") != "baseline":
        raise ValueError("Checkpoint is not the baseline L12 result")
    return files, metrics, status


def pf_current_screen(eq, field, original_currents, grid) -> dict:
    """Necessary-condition screen with boundary/pressure profile held fixed.

    Normal field is affine in PF currents at a fixed equilibrated surface. Fit
    that response in the exact DESC boundary objective, then verify the fitted
    vectors by direct evaluation. This is not a new equilibrium calculation.
    """
    import numpy as np
    from scipy.optimize import lsq_linear
    from desc.objectives import BoundaryError

    nominal = np.array([-125000.0, 19123.4976703, 19904.2740987,
                        -19395.6036404, 19304.7552541, 19123.4976703,
                        19904.2740987])
    k0 = original_currents[1:] / nominal[1:]
    obj = BoundaryError(eq=eq, field=field, field_fixed=False,
                        source_grid=grid, eval_grid=grid)
    obj.build(use_jit=False, verbose=0)
    n = int(obj.dim_f) // 2
    bn_scale = float(np.asarray(obj.normalization[0]))
    pressure_scale = float(np.asarray(obj.normalization[n]))

    def evaluate(relative_pf):
        field.currents = np.r_[nominal[0], nominal[1:] * relative_pf]
        f = np.asarray(obj.compute_unscaled(*obj.xs(eq, field)))
        if f.size != 2 * n or not np.isfinite(f).all():
            raise ValueError("Nonfinite or incomplete PF response")
        return f

    def describe(relative_pf, values):
        bn, pressure = values[:n], values[n:]
        return {
            "relative_pf_currents": np.asarray(relative_pf).tolist(),
            "boundary_normal_max_normalized": float(np.max(np.abs(bn)) / bn_scale),
            "boundary_normal_rms_normalized": float(np.sqrt(np.mean(bn ** 2)) / bn_scale),
            "boundary_pressure_max_normalized": float(np.max(np.abs(pressure)) / pressure_scale),
        }

    try:
        baseline = evaluate(k0)
        step = 0.1
        response = np.empty((n, 6))
        for j in range(6):
            trial = k0.copy()
            trial[j] += step
            response[:, j] = (evaluate(trial)[:n] - baseline[:n]) / step
        check = np.array([0.74, 1.1, 0.6, 0.88, 1.22, 0.95])
        linear_error = np.max(np.abs(evaluate(check)[:n] -
                                     (baseline[:n] + response @ (check - k0)))) / bn_scale
        if linear_error > 1e-9:
            raise ValueError(f"PF linear superposition failed: {linear_error}")
        result = {
            "scope": "Exact MGRID at the saved L16 force-balanced equilibrium and fixed surface; PF-only normal-field L2 fit, no new equilibrium solve",
            "baseline": describe(k0, baseline),
            "normal_field_linearity_max_normalized_error": float(linear_error),
        }
        theoretical, _, rank, _ = np.linalg.lstsq(
            response / bn_scale, (response @ k0 - baseline[:n]) / bn_scale,
            rcond=None,
        )
        floor = baseline[:n] + response @ (theoretical - k0)
        result["unbounded_mathematical_floor"] = {
            "relative_pf_currents": theoretical.tolist(),
            "boundary_normal_max_normalized": float(np.max(np.abs(floor)) / bn_scale),
            "boundary_normal_rms_normalized": float(np.sqrt(np.mean(floor ** 2)) / bn_scale),
            "response_rank": int(rank),
            "physical_proposal": False,
        }
        for name, lower in (("original_50_to_150_percent", 0.5),
                            ("diagnostic_0_to_150_percent", 0.0)):
            fit = lsq_linear(response / bn_scale,
                             (response @ k0 - baseline[:n]) / bn_scale,
                             bounds=(np.full(6, lower), np.full(6, 1.5)),
                             tol=1e-12, max_iter=1000)
            value = describe(fit.x, evaluate(fit.x))
            value["fit_succeeded"] = bool(fit.success)
            result[name] = value
        print("PF-only screen:", json.dumps(result, indent=2), flush=True)
        return result
    finally:
        field.currents = original_currents


def run_audit(mgrid: Path, files: dict[str, Path], saved: dict, status: dict) -> dict:
    import numpy as np
    from desc.equilibrium import Equilibrium
    from desc.grid import LinearGrid
    from desc.magnetic_fields import SplineMagneticField
    from desc.objectives import BoundaryError

    eqs = {
        "L12_free": Equilibrium.load(str(files["l12_free"])),
        "L16_fixed": Equilibrium.load(str(files["l16_fixed"])),
    }
    currents = np.asarray(saved["currents_A"], dtype=float)
    if currents.shape != (7,) or not np.isfinite(currents).all():
        raise ValueError("Missing or nonfinite seven-group PF/TF current vector")
    if not np.isclose(currents[0], -125000.0, rtol=0, atol=1e-6):
        raise ValueError("The TF current is not the saved 16 T setting")
    field = SplineMagneticField.from_mgrid(str(mgrid), extcur=currents.tolist(), method="linear")

    for key, eq in eqs.items():
        if int(eq.NFP) != 16 or not np.isclose(float(eq.Psi), 336.6068485660449, rtol=0, atol=1e-5):
            raise ValueError(f"{key} has the wrong field period or toroidal flux")

    common_boundary = LinearGrid(rho=np.array([1.0]), M=18, N=7, NFP=16, sym=False)
    shape_grid = LinearGrid(
        rho=np.array([1.0]), theta=np.linspace(0, 2 * np.pi, 128, endpoint=False),
        zeta=np.linspace(0, 2 * np.pi / 16, 32, endpoint=False), NFP=16
    )
    b12 = eqs["L12_free"].compute(["R", "Z"], grid=shape_grid)
    b16 = eqs["L16_fixed"].compute(["R", "Z"], grid=shape_grid)
    displacement = np.hypot(np.asarray(b12["R"]) - np.asarray(b16["R"]),
                            np.asarray(b12["Z"]) - np.asarray(b16["Z"]))
    shape = {
        "rms_displacement_m": float(np.sqrt(np.mean(displacement ** 2))),
        "max_displacement_m": float(np.max(displacement)),
        "same_surface_within_1mm": bool(np.max(displacement) < 1e-3),
    }

    result = {
        "method": "Read-only comparison of two saved equilibria at the same PF/TF currents",
        "desc_version": importlib.metadata.version("desc-opt"),
        "mgrid_sha256": MGRID_SHA256,
        "source_manifest_sha256": status["source_manifest_sha256"],
        "boundary_grid": {"M": 18, "N": 7, "NFP": 16, "sym": False},
        "currents_A": currents.tolist(),
        "surface_comparison": shape,
        "saved_L12_free_metrics": saved,
        "states": {},
    }

    for name, eq in eqs.items():
        print(f"Evaluating {name} on common boundary and volume grids", flush=True)
        obj = BoundaryError(eq=eq, field=field, field_fixed=False,
                            source_grid=common_boundary, eval_grid=common_boundary)
        obj.build(use_jit=False, verbose=0)
        capture = io.StringIO()
        with contextlib.redirect_stdout(capture):
            values = obj.print_value(obj.xs(eq, field))
        print(capture.getvalue(), flush=True)
        keys = ("Boundary normal field error: ", "Boundary magnetic pressure error: ")
        normal, pressure = (float(np.asarray(values[k]["f_max_norm"])) for k in keys)
        if not np.isfinite([normal, pressure]).all():
            raise ValueError(f"Nonfinite boundary error for {name}")
        volume = LinearGrid(rho=np.linspace(.05, .98, 10), M=20, N=6, NFP=16)
        force = np.asarray(eq.compute("|F|_normalized", grid=volume)["|F|_normalized"])
        if not np.isfinite(force).all():
            raise ValueError(f"Nonfinite force density for {name}")
        result["states"][name] = {
            "boundary_normal_max_normalized": normal,
            "boundary_pressure_max_normalized": pressure,
            "volume_force_normalized_rms_common_grid": float(np.sqrt(np.mean(force ** 2))),
            "volume_force_normalized_max_common_grid": float(np.max(force)),
            "boundary_gates_pass_on_common_grid": bool(normal <= BOUNDARY_GATE and pressure <= BOUNDARY_GATE),
            "volume_force_gate_pass_on_common_grid": bool(np.sqrt(np.mean(force ** 2)) <= FORCE_GATE),
        }

    measured = result["states"]["L12_free"]
    expected = saved["boundary_normal_desc"]["f_max_norm"]
    expected_p = saved["boundary_pressure_desc"]["f_max_norm"]
    matching = abs(measured["boundary_normal_max_normalized"] / expected - 1) < 0.05
    matching &= abs(measured["boundary_pressure_max_normalized"] / expected_p - 1) < 0.05
    result["L12_boundary_reproduction_within_5_percent"] = bool(matching)
    restored = result["states"]["L16_fixed"]
    if not matching:
        decision = "INCOMPARABLE: L12 published boundary errors were not reproduced."
    elif not shape["same_surface_within_1mm"]:
        decision = "SURFACE_CHANGED: L16 fixed solve changed the boundary; cannot isolate rebalance."
    elif restored["boundary_gates_pass_on_common_grid"] and restored["volume_force_gate_pass_on_common_grid"]:
        decision = "CANDIDATE: force and both boundary gates pass on this grid; verify at higher resolution."
    else:
        decision = "L12_UNRESOLVED: force rebalance alone did not pass all common-grid gates."
    result["decision"] = decision
    if matching and shape["same_surface_within_1mm"]:
        result["pf_only_screen"] = pf_current_screen(
            eqs["L16_fixed"], field, currents, common_boundary
        )
    result["G2_approved"] = False
    print(decision, flush=True)
    return result


def main() -> None:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    report = None
    try:
        if not INPUT_ROOT.is_dir():
            raise FileNotFoundError("Run this notebook on Kaggle with both input datasets attached")
        ensure_desc()
        files, saved, status = prepare_checkpoint()
        mgrid = prepare_mgrid()
        report = run_audit(mgrid, files, saved, status)
        json_write("L12_rebalance_audit.json", report)
    except Exception as error:
        report = {"G2_approved": False, "error": str(error), "traceback": traceback.format_exc()}
        json_write("audit_error.json", report)
        print(traceback.format_exc(), flush=True)
    finally:
        # The result ZIP is small: it contains metrics only, never the 425 MB MGRID.
        report_name = "audit_error.json" if report and "error" in report else "L12_rebalance_audit.json"
        with zipfile.ZipFile(OUT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
            path = OUT_DIR / report_name
            if path.is_file():
                archive.write(path, arcname=report_name)
        print(f"Download result: {OUT_ZIP}", flush=True)
    if report and "error" in report:
        raise RuntimeError(report["error"])


if __name__ == "__main__":
    main()
